# Problem 1: Bayesian Network Learning
## Naive Bayes

In [ ]:
# Naive Bayes Classifier
import numpy as np
import pandas as pd

class NaiveBayesClassifier():

    def __init__(self, smooth=0):
        self.smooth = smooth

    # Train model
    def fit(self, data):
        self.labels = data.iloc[:, 0].unique()
        self.feature_cols = range(1, data.shape[1])

        # Calculate priors for each label
        self.priors = data.iloc[:, 0].value_counts(normalize=True)

        # Calculate likelihood for each feature
        self.likelihood = {}
        for col in self.feature_cols:
            self.likelihood[col] = pd.crosstab(data[col], data[0])
            self.likelihood[col] += self.smooth
            self.likelihood[col] /= self.likelihood[col].sum(axis=0)

    # Test model
    def predict(self, sample):
        posterior = self.priors.copy()

        # Bayes theorem
        for col in self.feature_cols:
            for label in self.labels:
                posterior[label] *= self.likelihood[col][label][sample[col]]

        # Predict the label with max posterior
        return posterior.idxmax()

### No Regularization/Prior

In [2]:
train = pd.read_csv('mushroom_train.data', header=None)
test = pd.read_csv('mushroom_test.data', header=None)

model = NaiveBayesClassifier()
model.fit(train)

# Make a prediction for each sample in the test set
pred = test.apply(model.predict, axis=1)
actual = test.iloc[:, 0]

correct = (actual == pred).sum()
print(f"Correct: {correct}")
print(f"Wrong: {test.shape[0] - correct}")
print(f"Accuracy: {correct / test.shape[0]:.4f}")

Correct: 2640
Wrong: 5
Accuracy: 0.9981


#### What is your accuracy on mushroom test.data?
The accuracy is 99.81%
#### Is the performance reasonable?
This performance is reasonable for this particular dataset. It appears to be highly separable based on pairs of features. In the next few cells, you can see how most joint probabilities have one or two pairs with significantly higher probability than every other pair. Therefore, it only takes a few of these pairs in a sample to strongly push the probability toward one of the class labels. In other models, this might mean the model is overfitting the training set, however in this case the model's performance on the training set indicates that it is appropriately learning the function for predicting edibility.
#### Explain how to improve performance using a prior.
We can try to improve performance by bringing in some prior information that we would like the model to consider.
For this problem, I tested Laplace smoothing. We know that there may be some feature pairs in the test set that didn't show up in the training set.
The motivation behind Laplace smoothing is to avoid assigning zero probability to these feature pairs when they may still be valid.
To accomplish this, we start every pair with a count of 1 rather than 0. Therefore, after normalization, every joint probability will have at least a small value assigned to it.

In [3]:
model.likelihood[1]

,e,p
1,,
b,0.075431,0.017498
c,0.000000,0.003500
f,0.416487,0.447944
k,0.007004,0.008749
s,0.009159,0.000000
x,0.491918,0.522310


In [4]:
model.likelihood[5]

,e,p
5,,
a,0.125000,0.000000
c,0.000000,0.084864
f,0.000000,0.744532
l,0.113685,0.000000
m,0.000000,0.013998
n,0.761315,0.041995
p,0.000000,0.114611


In [5]:
model.likelihood[10]

,e,p
10,,
e,0.272091,0.864392
t,0.727909,0.135608


### With Regularization/Prior
The model actually performs worse after adding Laplace Smoothing as a prior. With a 99.8% accuracy, the model without a prior is almost exactly representing the true function for classifying mushrooms by edibility. Adding Laplace Smoothing generally helps when there are samples in the test set that didn't occur in the training set, and are therefore assigned zero probability. However, the model is clearly already recognizing every important pair in the test set when it achieves a 99.8% accuracy without smoothing.

In [6]:
train = pd.read_csv('mushroom_train.data', header=None)
test = pd.read_csv('mushroom_test.data', header=None)

model = NaiveBayesClassifier(smooth=1)
model.fit(train)

# Make a prediction for each sample in the test set
pred = test.apply(model.predict, axis=1)
actual = test.iloc[:, 0]

correct = (actual == pred).sum()
print(f"Correct: {correct}")
print(f"Wrong: {test.shape[0] - correct}")
print(f"Accuracy: {correct / test.shape[0]:.4f}")

Correct: 2526
Wrong: 119
Accuracy: 0.9550


## Structure Learning
### Chow-Liu Algorithm

In [7]:
# Kruskal's algorithm
from queue import PriorityQueue

def kruskal(V, E):
    F = set()
    forest = [{v} for v in V]

    find_set = lambda x: [tree for tree in forest if x in tree][0]

    q = PriorityQueue()
    for edge, weight in E.items():
        q.put((-weight, edge))

    while not q.empty():
        (_, (u,v)) = q.get()

        if find_set(u) != find_set(v):
            F.add((u,v))
            forest.append(find_set(u) | find_set(v))
            forest.remove(find_set(u))
            forest.remove(find_set(v))
    return F

In [8]:
# Chow-Liu
from collections import defaultdict
import numpy as np
import pandas as pd

def chow_liu(data):
    n = data.shape[1]

    marginals = {i: data[i].value_counts(normalize=True) for i in range(n)}

    edges = [(i,j) for i in range(n) for j in range(i+1, n)]

    MI = defaultdict(int)

    for (i,j) in edges:
        joint_prob = data.iloc[:, [i,j]].value_counts(normalize=True)

        for (x_i, x_j), P_ij in joint_prob.items():
            P_i = marginals[i][x_i]
            P_j = marginals[j][x_j]

            MI[(i,j)] += P_ij * np.log(P_ij / (P_i * P_j))

    return kruskal(range(n), MI)

In [9]:
# Convert an MRF to a Bayesian Network
def mrf_to_bn(mrf, root):
    parents = [root]
    bn = []

    while len(parents) > 0:
        parent = parents.pop(0)

        for (i,j) in [edge for edge in mrf if parent in edge]:
            mrf.remove((i,j))
            if j == parent:
                i,j = j,i
            bn.append((i,j))
            parents.append(j)
    return bn

In [10]:
train = pd.read_csv('mushroom_train.data', header=None)

mrf = chow_liu(train)
bn = mrf_to_bn(mrf, 0)

print("Learned Structure")
print(f"{'Parent':^8}| {'Child'}")
print('-' * 18)
for (i, j) in bn:
    print(f"{i:^8}|  {j:<5}")

Learned Structure
 Parent | Child
------------------
   0    |  5    
   0    |  16   
   5    |  13   
   5    |  10   
   5    |  9    
   5    |  3    
   5    |  11   
   5    |  8    
   5    |  20   
   13   |  19   
   11   |  1    
   11   |  22   
   11   |  21   
   20   |  18   
   19   |  12   
   19   |  14   
   19   |  7    
   19   |  15   
   19   |  4    
   21   |  2    
   18   |  6    
   14   |  17   


In [11]:
class BayesianClassifier():
    def __init__(self, bn, root, smooth=0):
        self.bn = bn
        self.root = root
        self.smooth = smooth

    def fit(self, data):
        # Class labels
        self.labels = train.iloc[:, self.root].unique()

        # Calculate priors for each class label
        self.priors = train.iloc[:, self.root].value_counts(normalize=True)

        # Conditional Probability Tables
        self.cpts = {}
        for (i,j) in bn:
            self.cpts[(i,j)] = pd.crosstab(train[j], train[i])
            self.cpts[(i,j)] += self.smooth
            self.cpts[(i,j)] /= self.cpts[(i,j)].sum(axis=0)

    def predict(self, sample):
        posterior = self.priors.copy()

        # Bayes theorem
        for (i,j) in self.bn:
            for label in self.labels:
                x_i = label if i == self.root else sample[i]
                x_j = label if j == self.root else sample[j]
                posterior[label] *= self.cpts[(i,j)][x_i][x_j]

        # Predict the label with max posterior
        return posterior.idxmax()

### No Regularization/Prior

In [12]:
train = pd.read_csv('mushroom_train.data', header=None)
test = pd.read_csv('mushroom_test.data', header=None)

root = 0

mrf = chow_liu(train)
bn = mrf_to_bn(mrf, root)

classifier = BayesianClassifier(bn, root)
classifier.fit(train)

# Make a prediction for each sample in the test set
pred = test.apply(classifier.predict, axis=1)
actual = test.iloc[:, root]

correct = (actual == pred).sum()
print(f"Correct: {correct}")
print(f"Wrong: {test.shape[0] - correct}")
print(f"Accuracy: {correct / test.shape[0]:.4f}")

Correct: 2605
Wrong: 40
Accuracy: 0.9849


### With Regularization/Prior

In [13]:
# Smoothing Factor of 1
train = pd.read_csv('mushroom_train.data', header=None)
test = pd.read_csv('mushroom_test.data', header=None)

root = 0

mrf = chow_liu(train)
bn = mrf_to_bn(mrf, root)

classifier = BayesianClassifier(bn, root, smooth=1)
classifier.fit(train)

# Make a prediction for each sample in the test set
pred = test.apply(classifier.predict, axis=1)
actual = test.iloc[:, root]

correct = (actual == pred).sum()
print(f"Correct: {correct}")
print(f"Wrong: {test.shape[0] - correct}")
print(f"Accuracy: {correct / test.shape[0]:.4f}")

Correct: 2605
Wrong: 40
Accuracy: 0.9849


## Performance Evaluation
### Naive Bayes
The Naive Bayes performs superbly without regularization, achieving 99.8% accuracy. Adding regularization actually decreases it's performance to 95.5% because it was already an extremely close fit to the underlying function without regularization.

### Learned Structure (Chow-Liu)
The structure learned by the Chow-Liu algorithm performed exactly the same with and without regularization at 98.49% accuracy. It is somewhat surprising that the Naive Bayes performed better. If the Naive Bayes structure was optimal, then I'd imagine the Chow-Liu algorithm would predict the Naive Bayes structure rather than the structure that was actually predicted.

The model learned by the Chow-Liu algorithm is more stable because it performs the same regardless of regularization. In fact, the model performs exactly the same all the way up to a smoothing factor of 96. This indicates that the joint distributions have much stronger probability separation in the learned structure than in the Naive Bayes structure.

In [14]:
# Smoothing Factor of 96
train = pd.read_csv('mushroom_train.data', header=None)
test = pd.read_csv('mushroom_test.data', header=None)

root = 0

mrf = chow_liu(train)
bn = mrf_to_bn(mrf, root)

classifier = BayesianClassifier(bn, root, smooth=96)
classifier.fit(train)

# Make a prediction for each sample in the test set
pred = test.apply(classifier.predict, axis=1)
actual = test.iloc[:, root]

correct = (actual == pred).sum()
print(f"Correct: {correct}")
print(f"Wrong: {test.shape[0] - correct}")
print(f"Accuracy: {correct / test.shape[0]:.4f}")

Correct: 2605
Wrong: 40
Accuracy: 0.9849
